In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")

# Structured output
Structured output allows agents to return data in a specific, predictable format. Instead of parsing natural language responses, you get structured data in the form of JSON objects, Pydantic models, or dataclasses that your application can use directly.

LangChain’s create_agent handles structured output automatically. The user sets their desired structured output schema, and when the model generates the structured data, it’s captured, validated, and returned in the 'structured_response' key of the agent’s state.

```python
def create_agent(
    ...
    response_format: Union[
        ToolStrategy[StructuredResponseT],
        ProviderStrategy[StructuredResponseT],
        type[StructuredResponseT],
        None,
    ]

## Response format
Use response_format to control how the agent returns structured data:
### 1: ToolStrategy[StructuredResponseT]: Uses tool calling for structured output
### 2: ProviderStrategy[StructuredResponseT]: Uses provider-native structured output
### 3: type[StructuredResponseT]: Schema type - automatically selects best strategy based on model capabilities
### 4: None: Structured output not explicitly requested
When a schema type is provided directly, LangChain automatically chooses:
ProviderStrategy if the model and provider chosen supports native structured output (e.g. OpenAI, Anthropic (Claude), or xAI (Grok)).
ToolStrategy for all other models.

## Provider strategy
Some model providers support structured output natively through their APIs (e.g. OpenAI, xAI (Grok), Gemini, Anthropic (Claude)). This is the most reliable method when available.
To use this strategy, configure a ProviderStrategy:

```python
class ProviderStrategy(Generic[SchemaT]):
    schema: type[SchemaT]
    strict: bool | None = None
    


# Schema for Structured Output

The **`schema`** parameter defines the structure of the output returned by the model or agent.  
It ensures that the response follows a **predictable and validated format** instead of plain text.

---

## Required

**`schema`** – The schema that defines the structured output format.

It supports the following types:

### 1. Pydantic Models
- Uses `BaseModel` subclasses.
- Supports field validation and type enforcement.
- Returns a **validated Pydantic instance**.

Example:

```python
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int

In [1]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain_anthropic import ChatAnthropic


class ContactInfo(BaseModel):
    """Contact information for a person."""

    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")


model = ChatAnthropic(
    model="claude-haiku-4-5",
    temperature=0.1,
    max_tokens_to_sample=100,
    timeout=30,
    max_retries=3,
)

agent = create_agent(
    model=model, response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567",
            }
        ]
    }
)

print(result["structured_response"])
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

name='John Doe' email='john@example.com' phone='(555) 123-4567'


### 2. Dataclasses
- Uses Python `dataclasses` with type annotations.
- Useful for lightweight structured data.

Example:

```python
from dataclasses import dataclass

@dataclass
class Product:
    name: str
    price: float

In [4]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain_anthropic import ChatAnthropic


@dataclass
class ContactInfo:
    """Contact information for a person."""

    name: str  # The name of the person
    email: str  # The email address of the person
    phone: str  # The phone number of the person


model = ChatAnthropic(
    model="claude-haiku-4-5",
    temperature=0.1,
    max_tokens_to_sample=100,
    timeout=30,
    max_retries=3,
)


agent = create_agent(
    model=model,
    response_format=ContactInfo,  # Auto-selects ProviderStrategy
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567",
            }
        ]
    }
)

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

### 3. TypedDict
- Uses `TypedDict` from the `typing` module.
- Enforces key and value types for dictionaries.

Example:

```python
from typing import TypedDict

class Order(TypedDict):
    order_id: int
    item: str

In [5]:
from typing_extensions import TypedDict
from langchain.agents import create_agent
from langchain_anthropic import ChatAnthropic


class ContactInfo(TypedDict):
    """Contact information for a person."""

    name: str  # The name of the person
    email: str  # The email address of the person
    phone: str  # The phone number of the person


model = ChatAnthropic(
    model="claude-haiku-4-5",
    temperature=0.1,
    max_tokens_to_sample=100,
    timeout=30,
    max_retries=3,
)

agent = create_agent(
    model=model,
    response_format=ContactInfo,  # Auto-selects ProviderStrategy
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567",
            }
        ]
    }
)

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

### 4. JSON Schema
- Uses a standard `JSON` Schema dictionary.
- Useful when integrating with APIs or external systems.

Example:

```python
schema = {
  "type": "object",
  "properties": {
    "title": {"type": "string"},
    "year": {"type": "integer"}
  },
  "required": ["title", "year"]
}

In [10]:
from langchain.agents import create_agent
from langchain_anthropic import ChatAnthropic
from langchain.agents.structured_output import ProviderStrategy


contact_info_schema = {
    "type": "object",
    "description": "Contact information for a person.",
    "properties": {
        "name": {"type": "string", "description": "The name of the person"},
        "email": {"type": "string", "description": "The email address of the person"},
        "phone": {"type": "string", "description": "The phone number of the person"},
    },
    "required": ["name", "email", "phone"],
}


model = ChatAnthropic(
    model="claude-haiku-4-5",
    temperature=0.1,
    max_tokens_to_sample=100,
    timeout=30,
    max_retries=3,
)

agent = create_agent(model=model, response_format=ProviderStrategy(contact_info_schema))

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567",
            }
        ]
    }
)

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

# Tool Calling Strategy (Structured Output)

For models that **don’t support native structured output**, LangChain uses a **tool calling strategy** to generate structured responses.

This approach works with **any model that supports tool calling**, which includes most modern LLMs.

To enable this behavior, configure a **`ToolStrategy`**.

---

## ToolStrategy Class

```python
class ToolStrategy(Generic[SchemaT]):
    schema: type[SchemaT]
    tool_message_content: str | None
    handle_errors: Union[
        bool,
        str,
        type[Exception],
        tuple[type[Exception], ...],
        Callable[[Exception], str],
    ]

# ToolStrategy Parameters

The **`ToolStrategy`** configuration defines how structured output should be generated and validated when using tool calling.

---

# 1. schema (required)

Defines the **structured output format** the model must return.

## Supported Schema Types

---

## Pydantic Models

- Uses **`BaseModel`** subclasses.
- Provides **strong validation and type checking**.
- Returns a **validated Pydantic instance**.

### Example

```python
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int

In [12]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_anthropic import ChatAnthropic


class ProductReview(BaseModel):
    """Analysis of a product review."""

    rating: int | None = Field(description="The rating of the product", ge=1, le=5)
    sentiment: Literal["positive", "negative"] = Field(
        description="The sentiment of the review"
    )
    key_points: list[str] = Field(
        description="The key points of the review. Lowercase, 1-3 words each."
    )


agent = create_agent(model=model, tools=[], response_format=ToolStrategy(ProductReview))

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Analyze this review: 'Great product: 5 out of 5 stars. Fast shipping, but expensive'",
            }
        ]
    }
)
result["structured_response"]
# ProductReview(rating=5, sentiment='positive', key_points=['fast shipping', 'expensive'])

ProductReview(rating=5, sentiment='positive', key_points=['fast shipping', 'expensive', 'great product'])

## Dataclasses

- Uses Python `dataclasses`.
- Lightweight structured data definition.

### Example

```python
from dataclasses import dataclass

@dataclass
class Product:
    name: str
    price: float

In [13]:
from dataclasses import dataclass
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


@dataclass
class ProductReview:
    """Analysis of a product review."""

    rating: int | None  # The rating of the product (1-5)
    sentiment: Literal["positive", "negative"]  # The sentiment of the review
    key_points: list[str]  # The key points of the review


agent = create_agent(model=model, tools=[], response_format=ToolStrategy(ProductReview))

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Analyze this review: 'Great product: 5 out of 5 stars. Fast shipping, but expensive'",
            }
        ]
    }
)
result["structured_response"]
# {'rating': 5, 'sentiment': 'positive', 'key_points': ['fast shipping', 'expensive']}

ProductReview(rating=5, sentiment='positive', key_points=['Great product', '5 out of 5 stars', 'Fast shipping', 'Expensive'])

## TypedDict

- Uses `TypedDict` from the `typing` module.
- Ensures dictionary **keys and value types**.

### Example

```python
from typing import TypedDict

class Order(TypedDict):
    order_id: int
    item: str

In [14]:
from typing import Literal
from typing_extensions import TypedDict
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ProductReview(TypedDict):
    """Analysis of a product review."""

    rating: int | None  # The rating of the product (1-5)
    sentiment: Literal["positive", "negative"]  # The sentiment of the review
    key_points: list[str]  # The key points of the review


agent = create_agent(model=model, tools=[], response_format=ToolStrategy(ProductReview))

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Analyze this review: 'Great product: 5 out of 5 stars. Fast shipping, but expensive'",
            }
        ]
    }
)
result["structured_response"]
# {'rating': 5, 'sentiment': 'positive', 'key_points': ['fast shipping', 'expensive']}

{'rating': 5,
 'sentiment': 'positive',
 'key_points': ['Great product',
  '5 out of 5 stars',
  'Fast shipping',
  'Expensive']}

## JSON Schema

- Uses **standard JSON Schema format**.
- Useful for **API integrations and schema validation.**

### Example

```python
schema = {
  "type": "object",
  "properties": {
    "title": {"type": "string"},
    "year": {"type": "integer"}
  },
  "required": ["title", "year"]
}

In [15]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


product_review_schema = {
    "type": "object",
    "description": "Analysis of a product review.",
    "properties": {
        "rating": {
            "type": ["integer", "null"],
            "description": "The rating of the product (1-5)",
            "minimum": 1,
            "maximum": 5,
        },
        "sentiment": {
            "type": "string",
            "enum": ["positive", "negative"],
            "description": "The sentiment of the review",
        },
        "key_points": {
            "type": "array",
            "items": {"type": "string"},
            "description": "The key points of the review",
        },
    },
    "required": ["sentiment", "key_points"],
}

agent = create_agent(
    model=model, tools=[], response_format=ToolStrategy(product_review_schema)
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Analyze this review: 'Great product: 5 out of 5 stars. Fast shipping, but expensive'",
            }
        ]
    }
)
result["structured_response"]
# {'rating': 5, 'sentiment': 'positive', 'key_points': ['fast shipping', 'expensive']}

{'sentiment': 'positive',
 'rating': 5,
 'key_points': ['Great product quality', 'Fast shipping', 'Expensive pricing']}

## Union Types

- Allows **multiple schema options**.
- The model automatically **selects the most appropriate schema** based on the context.

### Example

```python
from typing import Union
from pydantic import BaseModel

class Contact(BaseModel):
    name: str
    phone: str

class Email(BaseModel):
    name: str
    email: str

schema = Union[Contact, Email]

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ProductReview(BaseModel):
    """Analysis of a product review."""

    rating: int | None = Field(description="The rating of the product", ge=1, le=5)
    sentiment: Literal["positive", "negative"] = Field(
        description="The sentiment of the review"
    )
    key_points: list[str] = Field(
        description="The key points of the review. Lowercase, 1-3 words each."
    )


class CustomerComplaint(BaseModel):
    """A customer complaint about a product or service."""

    issue_type: Literal["product", "service", "shipping", "billing"] = Field(
        description="The type of issue"
    )
    severity: Literal["low", "medium", "high"] = Field(
        description="The severity of the complaint"
    )
    description: str = Field(description="Brief description of the complaint")


agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(Union[ProductReview, CustomerComplaint]),
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Analyze this review: 'Great product: 5 out of 5 stars. Fast shipping, but expensive'",
            }
        ]
    }
)
result["structured_response"]
# ProductReview(rating=5, sentiment='positive', key_points=['fast shipping', 'expensive'])

ProductReview(rating=5, sentiment='positive', key_points=['great product', 'fast shipping', 'expensive'])

In [18]:
result = agent.invoke(
    {
        "messages": [
            ("user", "Analyze this customer complaint: 'Product is too expensive'")
        ]
    }
)

structured_output = result["structured_response"]
structured_output

CustomerComplaint(issue_type='product', severity='medium', description='Product is too expensive')

# ToolStrategy Parameters

## 2. `tool_message_content`

Defines **custom content for the tool message** returned when structured output is generated.

If not provided, LangChain returns a **default message containing the structured response data**.

### Example

```python
from langchain.agents.structured_output import ToolStrategy

ToolStrategy(
    schema=User,
    tool_message_content="User data extracted successfully."
)

In [20]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class MeetingAction(BaseModel):
    """Action items extracted from a meeting transcript."""

    task: str = Field(description="The specific task to be completed")
    assignee: str = Field(description="Person responsible for the task")
    priority: Literal["low", "medium", "high"] = Field(description="Priority level")


agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(
        schema=MeetingAction,
        tool_message_content="Action item captured and added to meeting notes!",
    ),
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "From our meeting: Sarah needs to update the project timeline as soon as possible",
            }
        ]
    }
)   

result["structured_response"]

MeetingAction(task='Update the project timeline', assignee='Sarah', priority='high')

In [22]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "From our meeting: Sarah needs to update the project timeline as soon as possible",
            }
        ]
    }
)
result

{'messages': [HumanMessage(content='From our meeting: Sarah needs to update the project timeline as soon as possible', additional_kwargs={}, response_metadata={}, id='07b19f59-2162-4b31-beb5-6576f786cdf2'),
  AIMessage(content=[{'id': 'toolu_019gZRfoR3qRWDDXtpN4Nec8', 'caller': {'type': 'direct'}, 'input': {'task': 'Update the project timeline', 'assignee': 'Sarah', 'priority': 'high'}, 'name': 'MeetingAction', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_015veEW589abTkafDkTaktfV', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 734, 'output_tokens': 77, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'}, id='lc_run--01